In [17]:
# importing libraries

from scipy.io import loadmat
import os
import numpy as np
import pandas as pd




In [47]:
# loading a single mat file

data = loadmat('../../DataSets/Static_SOC/Static_SOC_1400_Points_3/Simulation_DC_FHDS_extended_SF_-2.50_SOC_H0.90_M0.80_L0.70_w1_0.01_w2_0.09_w3_0.61_wPL_0.000007_Sim757.mat')

In [48]:
print(data.keys())

# sample data keys
# dict_keys(['__header__', '__version__', '__globals__', 'D1', 'DC', 'H', 'I_b', 'I_cd', 'I_cells', 'J', 'L', 'M', 'PLoss', 'P_IO', 'S_factor', 'range', 'sim_idx', 'soc_M', 't_sim', 'v_t', 'w1', 'w2', 'w3', 'w_PL'])


dict_keys(['__header__', '__version__', '__globals__', 'D1', 'DC', 'H', 'I_b', 'I_cd', 'I_cells', 'J', 'L', 'M', 'PLoss', 'P_IO', 'S_factor', 'range', 'sim_idx', 'soc_M', 't_sim', 'v_t', 'w1', 'w2', 'w3', 'w_PL'])


In [49]:
# printing w1, w2, w3, wPL and range last value

print(data['w1'])
print(data['w2'])
print(data['w3'])
print(data['w_PL'])
print(data['DC'])
print(data['range'][-1])


[[0.01]]
[[0.09]]
[[0.61]]
[[6.97480923e-06]]
['FHDS_extended']
[213.31541667]


In [50]:
# Getting the list of all the files in the directory
files = os.listdir('../../DataSets/Static_SOC/Static_SOC_1400_Points_3')

# files filter only mat files
files = [file for file in files if '.mat' in file]

print(f"Total files: {len(files)}")

Total files: 1400


In [51]:
print(len(files))

1400


In [52]:
# Initialize an empty DataFrame
df = pd.DataFrame(columns=['w1', 'w2', 'w3', 'wPL', 'DC' , 'range'])

In [53]:
# List to collect rows for the DataFrame
rows = []

# Iterating over all the files and adding the values to the rows list
for file in files:
    file_path = os.path.join('../../DataSets/Static_SOC/Static_SOC_1400_Points_3', file)
    try:
        # Check if the file size is greater than zero
        if os.path.getsize(file_path) > 0:
            data = loadmat(file_path)
            # Check if all required keys are present in the file
            if 'w1' in data and 'w2' in data and 'w3' in data and 'w_PL' in data and 'range' in data and 'DC' in data:
                # Append a new row to the list
                rows.append({
                    'w1': data['w1'][0][0], 
                    'w2': data['w2'][0][0], 
                    'w3': data['w3'][0][0], 
                    'wPL': data['w_PL'][0][0], 
                    # adding DC which is a string 
                    'DC': data['DC'][0],
                    'range': data['range'][-1][0],
                })
            else:
                print(f"Skipping file with missing data: {file}")
        else:
            print(f"Skipping empty file: {file}")
    except Exception as e:
        print(f"Error reading {file}: {e}")

In [54]:
# Creating the DataFrame from the collected rows
df = pd.DataFrame(rows)

In [55]:
# Display the final DataFrame shape and check for duplicates
print(f"DataFrame shape: {df.shape}")
print(f"Number of unique rows: {df.drop_duplicates().shape[0]}")

DataFrame shape: (1400, 6)
Number of unique rows: 1400


In [56]:
# printing duplicate rows

print(df[df.duplicated()])


# printing files which are duplicate

print(df[df.duplicated(subset=['w1', 'w2', 'w3', 'wPL', 'DC', 'range'], keep=False)])




Empty DataFrame
Columns: [w1, w2, w3, wPL, DC, range]
Index: []
Empty DataFrame
Columns: [w1, w2, w3, wPL, DC, range]
Index: []


In [57]:
# Checking how many rows are there for each unique value of 'DC'
print(df['DC'].value_counts())

DC
FUDSfast_extended    200
FUDS_extended        200
NEDC_extended        200
US06City_extended    200
FHDS_extended        200
US06_extended        200
US06HW_extended      200
Name: count, dtype: int64


In [58]:
# dataframe without duplicates

df_without_duplicates = df.drop_duplicates()


In [59]:
print('DataFrames without duplicates', df_without_duplicates.shape)

print(df_without_duplicates['DC'].value_counts())


DataFrames without duplicates (1400, 6)
DC
FUDSfast_extended    200
FUDS_extended        200
NEDC_extended        200
US06City_extended    200
FHDS_extended        200
US06_extended        200
US06HW_extended      200
Name: count, dtype: int64


In [60]:
# Formatting the wPL column to %.6f
df['wPL'] = df['wPL'].map(lambda x: f"{x:.6f}")

# if directory does not exist then create it
if not os.path.exists('NewData'):
    os.makedirs('NewData')

# Saving the DataFrame to a CSV file
df.to_csv('NewData/Version_2_1400_3.csv', index=False)

# Display the resulting DataFrame
print(df)

        w1    w2    w3       wPL                 DC       range
0     0.10  0.14  0.86  0.000092  FUDSfast_extended  140.008833
1     0.10  0.48  0.29  0.000041      FUDS_extended  167.661833
2     0.02  0.09  0.51  0.000055      NEDC_extended  168.577083
3     0.12  0.59  0.62  0.000055      NEDC_extended  168.577083
4     0.33  0.03  0.69  0.000015      FUDS_extended  167.661833
...    ...   ...   ...       ...                ...         ...
1395  0.09  0.24  0.90  0.000023  FUDSfast_extended  140.098917
1396  0.66  0.50  0.63  0.000091      FHDS_extended  221.200694
1397  0.27  1.00  0.83  0.000090      FUDS_extended  167.661833
1398  0.35  0.90  0.01  0.000026  US06City_extended  170.073440
1399  0.63  0.53  0.19  0.000066      FUDS_extended  167.661833

[1400 rows x 6 columns]


In [61]:
df_700_1= pd.read_csv('NewData/Version_2_700_1.csv')
df_700_2= pd.read_csv('NewData/Version_2_700_2.csv')
df_700_3= pd.read_csv('NewData/Version_2_1400_3.csv')

In [62]:
# combining all the dataframes

df_combined = pd.concat([df_700_1, df_700_2, df_700_3])

In [63]:
# size of the combined dataframe

print(df_combined.shape)

(2800, 6)


In [64]:
# printing duplicate rows

print(df_combined[df_combined.duplicated()])


# printing files which are duplicate

print(df_combined[df_combined.duplicated(subset=['w1', 'w2', 'w3', 'wPL', 'DC', 'range'], keep=False)])

Empty DataFrame
Columns: [w1, w2, w3, wPL, DC, range]
Index: []
Empty DataFrame
Columns: [w1, w2, w3, wPL, DC, range]
Index: []


In [65]:
# saving the combined dataframe to a csv file

df_combined.to_csv('NewData/Version_2_Combined_DataSet.csv', index=False)